# Stage 3 Track B — cetacean detector (YOLOv8), v1

**Why this exists.** The end-to-end test showed that Track B classifies a cetacean box well when a person draws the box (20 of 22 images),
but nothing in the pipeline can *find* the box: Stage 1's region covers about 78% of the frame and SegFormer was never trained on cetaceans.
With Stage 1's region, Track B wrongly called 29% of tuna frames cetacean. A detector learns what a cetacean **looks like**, and also where it is.

**What this notebook does**
1. **Regroups the cetacean photos by picture content.** The old split grouped photos by box size only. The end-to-end audit found that
   94% of Track B's held-out photos had a near-identical copy (tinted, grey, noisy or resized) in training. Here every photo is fingerprinted
   and look-alikes are joined into one group, so no copy can sit on both sides of the split.
2. **Keeps the end-to-end test pictures out.** Any training picture that matches a `Stage_03/Test_data` picture is dropped.
3. **Adds negatives:** prawn, tuna and open-sea frames with no cetacean, so the detector learns what NOT to fire on.
4. **Trains YOLOv8n** on train / val, picks the confidence cut-off on **val**, and reports on **test** once: cetacean photos, plus
   prawn / tuna / sea frames from the held-out split and from `Stage_03/Test_data`.
5. Saves `stage3_output/cetacean_detector_v1.pt` and `cetacean_detector_v1.json` (cut-off, split lists, results).

**Where to run it**
- **Your own computer:** works as is (cell 1 uses your Windows project path). Install once: `pip install ultralytics`.
  With an NVIDIA GPU and a CUDA build of PyTorch it is fast. Without one, cell 5 switches to lighter CPU settings automatically;
  set `TRIAL_EPOCHS = 1` in cell 1 first to time one epoch, then set it back to `None` and run for real (expect several hours, e.g. overnight).
- **Google Colab (faster if you have no GPU):** copy the project folder to Google Drive, Runtime → Change runtime type → T4 GPU.

Run the cells in order, once.

**Licence note:** the `ultralytics` package is AGPL-3.0. That is fine for a university project; check with Saqqara before shipping it in a product.

In [1]:
# --- Cell 1: Settings ---
import os, sys, json, random, shutil, time
from pathlib import Path
import numpy as np
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Saqqara-Marine-Anomaly-Detection")   # <- change if your folder is elsewhere on Drive
    WORK_DIR = Path("/content/cetacean_yolo")                                        # local disk: much faster than Drive
else:
    PROJECT_ROOT = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection")
    WORK_DIR = PROJECT_ROOT / "Stage_03" / "cetacean_yolo"

S3 = PROJECT_ROOT / "Stage_03"
CET_IMAGES = S3 / "Stage3_TrackB" / "train" / "images"
CET_LABELS = S3 / "Stage3_TrackB" / "train" / "labels"

# Negatives: frames with NO cetacean. Only development folders - never Stage_03/Test_data.
# (Stage_01/data/prawns and /tuna are the same pictures as Test_data, and Stage_01/data/normal_sea/heldout_split is Test_data/SEA,
#  so they are deliberately NOT listed.)
NEGATIVE_DIRS = {
    "prawn": [PROJECT_ROOT / "Stage_02" / "prawn_annotation_training" / "images" / "default"],
    "tuna":  [PROJECT_ROOT / "Stage_02" / "tuna_annotation_training" / "images" / "default", S3 / "Val_data" / "TUNA"],
    "sea":   [S3 / "Val_data" / "SEA", PROJECT_ROOT / "Stage_01" / "data" / "normal_sea" / "baseline_split"],
}
MAX_NEGATIVES_PER_KIND = 400

# The end-to-end test pictures: never trained on; used only for the final false-alarm check.
TEST_DATA_DIRS = {"prawn": S3 / "Test_data" / "PRAWNS", "tuna": S3 / "Test_data" / "TUNA", "sea": S3 / "Test_data" / "SEA"}

# Cetacean photos whose label file is empty: unknown whether that means "no animal" or "not labelled yet".
# "exclude" (default, safe) leaves them out; "negative" would teach the detector that nothing is there.
EMPTY_LABEL_POLICY = "exclude"

# Grouping: two pictures are copies if their 32x32 grey thumbnails correlate at least this much (any flip).
GROUP_SIMILARITY = 0.99
LEAK_SIMILARITY = {"sea": 0.985, "default": 0.95}   # a training picture this close to a Test_data picture is dropped
# (open water looks alike at thumbnail size, so sea needs a higher bar - same values as the end-to-end audit)
SPLIT = (0.70, 0.15, 0.15)      # train / val / test, by group
SEED = 42

# Training
MODEL_WEIGHTS = "yolov8n.pt"
IMG_SIZE = 640                  # the animals are small; 640 keeps them visible
EPOCHS = 60
PATIENCE = 15
BATCH = 32
# Running on your own computer WITHOUT an NVIDIA GPU: these lighter settings are used automatically in cell 5.
CPU_IMG_SIZE = 352              # the photos are mostly 224-331 px, so 352 loses little
CPU_EPOCHS = 30
CPU_BATCH = 16
TRIAL_EPOCHS = None             # e.g. 1: train one epoch only, to see how long an epoch takes on your machine
CONF_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]

OUT_DIR = S3 / "stage3_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png"}
def list_images(folder):
    folder = Path(folder)
    return sorted(p for p in folder.glob("*") if p.suffix.lower() in IMG_EXTS) if folder.exists() else []

missing = [str(p) for p in [CET_IMAGES, CET_LABELS] if not p.exists()]
if missing:
    raise RuntimeError("Not found - fix PROJECT_ROOT in this cell:\n  " + "\n  ".join(missing))
print("Project:", PROJECT_ROOT)
print("Work folder:", WORK_DIR)

Project: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection
Work folder: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\cetacean_yolo


In [2]:
# --- Cell 2: Load cetacean photos and their boxes; list negatives ---

def read_yolo(txt):
    rows = []
    for line in Path(txt).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            rows.append(tuple(float(v) for v in parts[1:5]))      # xc, yc, w, h (0-1); every class becomes 'cetacean'
    return rows

label_of = {p.stem: p for p in CET_LABELS.glob("*.txt")}
cet = []          # dicts: path, boxes (normalised), kind
n_nolabel = n_empty = 0
for p in list_images(CET_IMAGES):
    t = label_of.get(p.stem)
    if t is None:
        n_nolabel += 1
        continue
    b = read_yolo(t)
    if not b:
        n_empty += 1
        if EMPTY_LABEL_POLICY != "negative":
            continue
    cet.append({"path": p, "boxes": b, "kind": "cetacean" if b else "cetacean_empty"})
print(f"Cetacean photos: {sum(r['kind'] == 'cetacean' for r in cet)} with boxes "
      f"({sum(len(r['boxes']) for r in cet)} animals); {n_empty} with an empty label file "
      f"({'used as negatives' if EMPTY_LABEL_POLICY == 'negative' else 'left out'}); {n_nolabel} with no label file (left out)")

neg = []
for kind, dirs in NEGATIVE_DIRS.items():
    paths = []
    for d in dirs:
        found = list_images(d)
        print(f"  negatives {kind:<6} {len(found):>5} in {d}" + ("   (FOLDER NOT FOUND)" if not Path(d).exists() else ""))
        paths += found
    paths = list(dict.fromkeys(paths))
    random.Random(SEED).shuffle(paths)
    neg += [{"path": p, "boxes": [], "kind": kind} for p in paths[:MAX_NEGATIVES_PER_KIND]]
print("Negatives kept:", {k: sum(r["kind"] == k for r in neg) for k in NEGATIVE_DIRS})

test_data = {k: list_images(d) for k, d in TEST_DATA_DIRS.items()}
print("Test_data pictures (never trained on):", {k: len(v) for k, v in test_data.items()})

Cetacean photos: 5632 with boxes (19850 animals); 1828 with an empty label file (left out); 0 with no label file (left out)
  negatives prawn     25 in C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_02\prawn_annotation_training\images\default
  negatives tuna      26 in C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_02\tuna_annotation_training\images\default
  negatives tuna     539 in C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\Val_data\TUNA
  negatives sea      916 in C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\Val_data\SEA
  negatives sea     4199 in C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\normal_sea\baseline_split
Negatives kept: {'prawn': 25, 'tuna': 400, 'sea': 400}
Test_data pictures (never trained on): {'prawn': 82, 'tuna': 236, 'sea': 457}


In [3]:
# --- Cell 3: Group look-alike pictures by content, and drop anything that matches the Test_data pictures ---

def thumb(path):
    with Image.open(path) as im:
        im.draft("L", (128, 128))
        g = np.asarray(im.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float32)
    def norm(a):
        a = a - a.mean(); n = np.linalg.norm(a)
        return (a / n).ravel() if n > 1e-6 else np.zeros(a.size, np.float32)
    return norm(g), np.stack([norm(g[:, ::-1]), norm(g[::-1, :]), norm(g[::-1, ::-1])])

def fingerprints(items):
    V, F, ok = [], [], []
    for r in items:
        try:
            v, f = thumb(r["path"]); V.append(v); F.append(f); ok.append(r)
        except Exception as e:
            print("  could not read", Path(r["path"]).name, e)
    return np.stack(V), np.stack(F), ok

t0 = time.time()
items = cet + neg
V, F, items = fingerprints(items)
print(f"Fingerprinted {len(items)} pictures ({time.time() - t0:.0f}s)")

# 1) leak guard: anything close to a Test_data picture is dropped
TV = np.stack([thumb(p)[0] for ps in test_data.values() for p in ps]) if any(test_data.values()) else None
test_names = {p.name for ps in test_data.values() for p in ps}
drop = np.zeros(len(items), bool)
if TV is not None:
    for s in range(0, len(items), 1024):
        sim = np.max(np.stack([V[s:s + 1024] @ TV.T] + [F[s:s + 1024, k] @ TV.T for k in range(3)]), axis=0).max(axis=1)
        thr = np.array([LEAK_SIMILARITY.get(r["kind"], LEAK_SIMILARITY["default"]) for r in items[s:s + 1024]])
        drop[s:s + 1024] = sim >= thr
for i, r in enumerate(items):
    drop[i] |= Path(r["path"]).name in test_names
print("Dropped as look-alikes of Test_data pictures:",
      {k: int(sum(drop[i] for i, r in enumerate(items) if r["kind"] == k)) for k in sorted({r["kind"] for r in items})})
items = [r for i, r in enumerate(items) if not drop[i]]
V, F = V[~drop], F[~drop]

# 2) union-find on look-alikes (any flip), within the remaining pictures
parent = list(range(len(items)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra
n_pairs = 0
for s in range(0, len(items), 1024):
    blk = np.max(np.stack([V[s:s + 1024] @ V.T] + [F[s:s + 1024, k] @ V.T for k in range(3)]), axis=0)
    ii, jj = np.nonzero(blk >= GROUP_SIMILARITY)
    for a, b in zip(ii + s, jj):
        if a < b:
            union(int(a), int(b)); n_pairs += 1
# copies that share exactly the same set of box sizes (rotated / flipped copies) are joined too
by_dims = {}
for i, r in enumerate(items):
    if r["boxes"]:
        key = tuple(sorted(tuple(sorted((round(w, 3), round(h, 3)))) for _, _, w, h in r["boxes"]))
        by_dims.setdefault(key, []).append(i)
for idx in by_dims.values():
    for j in idx[1:]:
        union(idx[0], j)
for i, r in enumerate(items):
    r["group"] = find(i)

sizes = np.bincount([r["group"] for r in items])
sizes = sizes[sizes > 0]
print(f"\n{len(items)} pictures -> {len(sizes)} groups  (look-alike pairs found: {n_pairs})")
print(f"  group size: median {int(np.median(sizes))}, 90th percentile {int(np.percentile(sizes, 90))}, largest {int(sizes.max())}")
for k in sorted({r["kind"] for r in items}):
    print(f"  {k:<15} {sum(r['kind'] == k for r in items):>6} pictures in {len({r['group'] for r in items if r['kind'] == k}):>5} groups")
if sizes.max() > 0.10 * len(items):
    print(f"\nWARNING: one group holds {sizes.max()} pictures ({100 * sizes.max() / len(items):.0f}%). Plain-water pictures may be chaining "
          "together. Raise GROUP_SIMILARITY (e.g. 0.995) and re-run this cell.")

Fingerprinted 6457 pictures (46s)
Dropped as look-alikes of Test_data pictures: {'cetacean': 442, 'prawn': 25, 'sea': 34, 'tuna': 89}

5867 pictures -> 982 groups  (look-alike pairs found: 715651)
  group size: median 1, 90th percentile 11, largest 2332
  cetacean          5190 pictures in   372 groups
  sea                366 pictures in   355 groups
  tuna               311 pictures in   255 groups



In [4]:
# --- Cell 4: Split by group into train / val / test, and write the YOLO dataset ---

groups = sorted({r["group"] for r in items})
random.Random(SEED).shuffle(groups)
n = len(groups)
cut1, cut2 = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
split_of_group = {g: ("train" if k < cut1 else "val" if k < cut2 else "test") for k, g in enumerate(groups)}
for r in items:
    r["split"] = split_of_group[r["group"]]
assert all(len({r["split"] for r in items if r["group"] == g}) == 1 for g in groups[:2000]), "a group was split - should never happen"

print(f"{'kind':<15}" + "".join(f"{s:>8}" for s in ("train", "val", "test")))
for k in sorted({r["kind"] for r in items}):
    print(f"{k:<15}" + "".join(f"{sum(r['kind'] == k and r['split'] == s for r in items):>8}" for s in ("train", "val", "test")))

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
for s in ("train", "val", "test"):
    (WORK_DIR / "images" / s).mkdir(parents=True)
    (WORK_DIR / "labels" / s).mkdir(parents=True)
for k, r in enumerate(items):
    name = f"{r['kind']}_{k:06d}{Path(r['path']).suffix.lower()}"          # unique names: different folders reuse filenames
    r["yolo_name"] = name
    shutil.copy2(r["path"], WORK_DIR / "images" / r["split"] / name)
    lines = [f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}" for xc, yc, w, h in r["boxes"]]
    (WORK_DIR / "labels" / r["split"] / (Path(name).stem + ".txt")).write_text("\n".join(lines))
yaml_path = WORK_DIR / "data.yaml"
yaml_path.write_text(f"path: {WORK_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: cetacean\n")
print(f"\nDataset written to {WORK_DIR}  ({len(items)} pictures)")

kind              train     val    test
cetacean           4335     413     442
sea                 244      62      60
tuna                225      50      36

Dataset written to C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\cetacean_yolo  (5867 pictures)


In [5]:
# --- Cell 5: Train YOLOv8n (GPU) ---
try:
    import ultralytics
except ImportError:
    os.system(f"{sys.executable} -m pip install -q ultralytics")
    import ultralytics
from ultralytics import YOLO
import torch
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - this will be very slow")

GPU = torch.cuda.is_available()
if not GPU:
    IMG_SIZE, EPOCHS, BATCH = CPU_IMG_SIZE, CPU_EPOCHS, CPU_BATCH      # later cells use the same IMG_SIZE
    print(f"No GPU: using the CPU settings (image size {IMG_SIZE}, up to {EPOCHS} epochs, batch {BATCH}). "
          "Expect hours - YOLO prints the time per epoch; stop and use Colab if one epoch takes more than ~30 minutes.")
if TRIAL_EPOCHS:
    EPOCHS = TRIAL_EPOCHS
    print(f"TRIAL RUN: {EPOCHS} epoch(s) only - the results below will be poor; this is just for timing.")
model = YOLO(MODEL_WEIGHTS)
train_res = model.train(data=str(yaml_path), imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, seed=SEED,
                        project=str(WORK_DIR / "runs"), name="cetacean_v1", exist_ok=True,
                        device=0 if GPU else "cpu", workers=8 if GPU else 0,   # workers=0: avoids hangs in Windows notebooks
                        plots=True, verbose=False)
BEST = Path(train_res.save_dir) / "weights" / "best.pt"
print("Best weights:", BEST)

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\bcura\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.178 | GPU: NONE - this will be very slow
No GPU: using the CPU settings (image size 352, up to 30 epochs, batch 16). Expect hours - YOLO prints the time per epoch; stop and use Colab if one epoch takes more than ~30 minutes.
Ultralytics 8.4.178  Python-3.14.6 torch-2.14.0+cpu CPU (AMD Ryzen 7 7435HS)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\bcura\AI PostGrad Project\Saqqar

In [6]:
# --- Cell 6: Run the trained detector on val, test and Test_data pictures (one pass; later cells only re-count) ---
det = YOLO(str(BEST))

def predict(paths, batch=32):
    out = {}
    for s in range(0, len(paths), batch):
        for res in det.predict([str(p) for p in paths[s:s + batch]], imgsz=IMG_SIZE, conf=min(CONF_GRID), verbose=False):
            b = res.boxes
            out[str(Path(res.path))] = {"conf": b.conf.cpu().numpy().tolist(), "xyxy": b.xyxy.cpu().numpy().tolist(),
                                        "wh": res.orig_shape[::-1]}
    return out

pred = {}
for s in ("val", "test"):
    paths = [WORK_DIR / "images" / s / r["yolo_name"] for r in items if r["split"] == s]
    pred.update(predict(paths))
td_paths = [p for ps in test_data.values() for p in ps]
pred_td = predict(td_paths)
print(f"Predicted {len(pred)} val/test pictures and {len(pred_td)} Test_data pictures.")

Predicted 1063 val/test pictures and 775 Test_data pictures.


In [7]:
# --- Cell 7: Pick the confidence cut-off on VAL (never on test) ---
# An image counts as 'cetacean' if any box is at or above the cut-off. A cetacean image counts as FOUND only if such a box
# overlaps a labelled animal (IoU >= 0.3), so firing on the wrong spot does not count.

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0

def gt_xyxy(r, wh):
    W, H = wh
    return [((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H) for xc, yc, w, h in r["boxes"]]

def image_outcome(r, p, conf):
    """('found' | 'missed') for cetacean pictures, ('false_alarm' | 'quiet') for the rest."""
    hits = [b for c, b in zip(p["conf"], p["xyxy"]) if c >= conf]
    if r["kind"] == "cetacean":
        g = gt_xyxy(r, p["wh"])
        return "found" if any(iou(h, x) >= 0.3 for h in hits for x in g) else "missed"
    return "false_alarm" if hits else "quiet"

def animal_recall(rs, conf):
    caught = total = 0
    for r in rs:
        p = pred[str(WORK_DIR / "images" / r["split"] / r["yolo_name"])]
        hits = [b for c, b in zip(p["conf"], p["xyxy"]) if c >= conf]
        for g in gt_xyxy(r, p["wh"]):
            total += 1; caught += any(iou(h, g) >= 0.3 for h in hits)
    return caught, total

def table(split):
    rs = [r for r in items if r["split"] == split]
    rows = []
    for conf in CONF_GRID:
        o = [(r["kind"], image_outcome(r, pred[str(WORK_DIR / "images" / split / r["yolo_name"])], conf)) for r in rs]
        pos = [x for k, x in o if k == "cetacean"]
        negs = [x for k, x in o if k != "cetacean"]
        rec = sum(x == "found" for x in pos) / max(1, len(pos))
        fa = sum(x == "false_alarm" for x in negs) / max(1, len(negs))
        rows.append((conf, rec, fa, {k: (sum(x == "false_alarm" for kk, x in o if kk == k), sum(kk == k for kk, _ in o))
                                     for k in sorted({k for k, _ in o if k != "cetacean"})}, len(pos)))
    return rows

print("VAL - cetacean images found vs other images falsely flagged, by cut-off")
val_rows = table("val")
for conf, rec, fa, per, npos in val_rows:
    print(f"  conf {conf:<5} cetacean found {100 * rec:5.1f}% of {npos}   false alarms {100 * fa:5.1f}%   " +
          "  ".join(f"{k} {a}/{b}" for k, (a, b) in per.items()))
# choose: best balanced score (found + not-false-alarm) / 2; ties -> higher cut-off
CONF = max(val_rows, key=lambda t: ((t[1] + (1 - t[2])) / 2, t[0]))[0]
print(f"\nChosen on VAL: confidence cut-off {CONF}")

VAL - cetacean images found vs other images falsely flagged, by cut-off
  conf 0.1   cetacean found  94.4% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.15  cetacean found  93.2% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.2   cetacean found  91.3% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.25  cetacean found  89.6% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.3   cetacean found  88.1% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.4   cetacean found  84.0% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.5   cetacean found  82.1% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.6   cetacean found  76.8% of 413   false alarms   0.0%   sea 0/62  tuna 0/50
  conf 0.7   cetacean found  66.3% of 413   false alarms   0.0%   sea 0/62  tuna 0/50

Chosen on VAL: confidence cut-off 0.1


In [8]:
# --- Cell 8: TEST - reported once, at the cut-off chosen on val ---
from scipy.stats import beta
def rate(k, n):
    if n == 0:
        return "n/a"
    lo = 0.0 if k == 0 else beta.ppf(0.025, k, n - k + 1); hi = 1.0 if k == n else beta.ppf(0.975, k + 1, n - k)
    return f"{k}/{n} = {100 * k / n:5.1f}%   (95% CI {100 * lo:.0f}-{100 * hi:.0f}%)"

test_rs = [r for r in items if r["split"] == "test"]
res = {"conf": CONF}
print(f"TEST SPLIT (groups never seen in training), cut-off {CONF}")
pos = [r for r in test_rs if r["kind"] == "cetacean"]
found = sum(image_outcome(r, pred[str(WORK_DIR / "images" / "test" / r["yolo_name"])], CONF) == "found" for r in pos)
c, t = animal_recall(pos, CONF)
print(f"  cetacean images found (box on an animal): {rate(found, len(pos))}")
print(f"  animals found:                            {rate(c, t)}")
res["test_cetacean_images"] = [found, len(pos)]; res["test_cetacean_animals"] = [c, t]
for k in sorted({r["kind"] for r in test_rs if r["kind"] != "cetacean"}):
    rs = [r for r in test_rs if r["kind"] == k]
    fa = sum(image_outcome(r, pred[str(WORK_DIR / "images" / "test" / r["yolo_name"])], CONF) == "false_alarm" for r in rs)
    print(f"  {k:<14} wrongly flagged:          {rate(fa, len(rs))}")
    res[f"test_false_alarm_{k}"] = [fa, len(rs)]

print(f"\nSTAGE_03/Test_data (the end-to-end test pictures; never trained on), cut-off {CONF}")
for k, ps in test_data.items():
    fa = sum(any(c >= CONF for c in pred_td[str(p)]["conf"]) for p in ps if str(p) in pred_td)
    print(f"  {k:<6} frames wrongly flagged as cetacean: {rate(fa, len(ps))}")
    res[f"test_data_false_alarm_{k}"] = [fa, len(ps)]

m = det.val(data=str(yaml_path), split="test", imgsz=IMG_SIZE, verbose=False, plots=False)
res["test_map50"] = float(m.box.map50); res["test_map50_95"] = float(m.box.map)
print(f"\nStandard detection scores on the test split: mAP@0.5 = {m.box.map50:.3f}, mAP@0.5:0.95 = {m.box.map:.3f}")
print("For comparison - Track B (colour/brightness) on Stage 1's region wrongly called 29% of tuna and 40% of prawn frames cetacean.")

TEST SPLIT (groups never seen in training), cut-off 0.1
  cetacean images found (box on an animal): 418/442 =  94.6%   (95% CI 92-96%)
  animals found:                            1531/1677 =  91.3%   (95% CI 90-93%)
  sea            wrongly flagged:          0/60 =   0.0%   (95% CI 0-6%)
  tuna           wrongly flagged:          0/36 =   0.0%   (95% CI 0-10%)

STAGE_03/Test_data (the end-to-end test pictures; never trained on), cut-off 0.1
  prawn  frames wrongly flagged as cetacean: 31/82 =  37.8%   (95% CI 27-49%)
  tuna   frames wrongly flagged as cetacean: 4/236 =   1.7%   (95% CI 0-4%)
  sea    frames wrongly flagged as cetacean: 9/457 =   2.0%   (95% CI 1-4%)
Ultralytics 8.4.178  Python-3.14.6 torch-2.14.0+cpu CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 716.8±165.6 MB/s, size: 243.3 KB)
val: Scanning C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Sta

In [9]:
# --- Cell 9: Save the detector and everything needed to reuse or audit it ---
dst = OUT_DIR / "cetacean_detector_v1.pt"
shutil.copy2(BEST, dst)
bundle = {
    "weights": dst.name, "confidence_cutoff": CONF, "chosen_on": "val split (image-level: found + not false alarm)",
    "img_size": IMG_SIZE, "model": MODEL_WEIGHTS, "epochs_max": EPOCHS,
    "grouping": {"method": "32x32 grey thumbnail correlation (any flip) + same box-size set", "similarity": GROUP_SIMILARITY,
                 "leak_similarity_vs_test_data": LEAK_SIMILARITY},
    "empty_label_policy": EMPTY_LABEL_POLICY,
    "files": {s: [{"source": str(r["path"]), "kind": r["kind"], "group": int(r["group"])} for r in items if r["split"] == s]
              for s in ("train", "val", "test")},
    "results": res,
    "val_table": [{"conf": c, "cetacean_found": rec, "false_alarm": fa} for c, rec, fa, _, _ in val_rows],
}
with open(OUT_DIR / "cetacean_detector_v1.json", "w") as f:
    json.dump(bundle, f, indent=1)
print("Saved:", dst)
print("Saved:", OUT_DIR / "cetacean_detector_v1.json")
print("The 'test' cetacean photos listed there are the ones the integrated end-to-end notebook should use next.")

Saved: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\stage3_output\cetacean_detector_v1.pt
Saved: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\stage3_output\cetacean_detector_v1.json
The 'test' cetacean photos listed there are the ones the integrated end-to-end notebook should use next.
